# 05 · Clasificador de envases — entrenamiento en Google Colab (GPU)

Mismo entrenamiento que `05_clasificador.ipynb`, pero en Colab con GPU. Corre **exactamente el
mismo** `scripts/clasificador_entrenar.py` (viaja dentro del zip), así que los resultados son
comparables y el `.keras` se usa tal cual en la PC.

**Antes (en la PC):**
```
conda activate inventariovision
python scripts/clasificador_empaquetar_colab.py empaquetar
```
→ genera `modelos/clasificador/colab/clasificador_colab.zip` (código + `datos/recortes`).

**En Colab:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*, y correr todas las celdas.
La celda 2 pide subir el zip. La última descarga `resultados_clasificador.zip`.

**Después (en la PC):**
```
python scripts/clasificador_empaquetar_colab.py importar %USERPROFILE%\Downloads\resultados_clasificador.zip
```
→ deja `modelos/clasificador/` (clasificador.keras, clases.json, metricas.json, comparacion.csv, …)
y `logs/clasificador/` (TensorBoard) en su lugar. Después se abre `05_clasificador.ipynb`
(secciones 8 a 11) para ver la comparación.

In [ ]:
# 1. GPU y versiones
!nvidia-smi -L || echo "SIN GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU"
import tensorflow as tf, keras
print('TensorFlow', tf.__version__, '· Keras', keras.__version__, '· GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 2. Subir el zip (o copiarlo desde Drive: ver la celda siguiente)
import zipfile, os
from pathlib import Path
from google.colab import files
ZIP = Path('/content/clasificador_colab.zip')
if not ZIP.exists():
    subido = files.upload()                       # elegir clasificador_colab.zip
    nombre = next(iter(subido))
    Path(nombre).rename(ZIP)

In [ ]:
# 2b. (Opcional) Si el zip está en Google Drive, en vez de subirlo:
# from google.colab import drive; drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/clasificador_colab.zip" /content/clasificador_colab.zip

In [ ]:
# 3. Descomprimir y preparar rutas (las mismas variables de entorno que usa src/config.py)
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content')
RAIZ = Path('/content/control_inventario')
os.environ['INVENTARIO_DATOS'] = str(RAIZ / 'datos')
os.environ['INVENTARIO_MODELOS'] = str(RAIZ / 'modelos')
os.environ['INVENTARIO_LOGS'] = str(RAIZ / 'logs')
os.chdir(RAIZ)
!pip -q install python-dotenv
n = sum(1 for _ in (RAIZ / 'datos' / 'recortes').rglob('*.jpg'))
print('Recortes:', n)

In [ ]:
# 4. Prueba corta (2 épocas, 16 recortes por clase): detecta errores y mide velocidad en GPU
!python -u scripts/clasificador_entrenar.py --modelo todos --prueba --hilos 0 2>&1 | grep --line-buffered -v -E "oneDNN|cpu_feature|UserWarning|data_adapter"

In [ ]:
# 5. Entrenamiento completo de los 3 modelos (cnn, cnn_ad, mobilenet). En T4: del orden de minutos por modelo.
#    Para uno solo: --modelo mobilenet. Para acortar: --epocas N --epocas-finetune M.
!python -u scripts/clasificador_entrenar.py --modelo todos --hilos 0 2>&1 | grep --line-buffered -v -E "oneDNN|cpu_feature|UserWarning|data_adapter"

In [ ]:
# 6. TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/control_inventario/logs/clasificador

In [ ]:
# 7. Comparación
import pandas as pd
from IPython.display import display, Image
comp = pd.read_csv(RAIZ / 'modelos/clasificador/comparacion.csv')
display(comp)
import json
m = json.loads((RAIZ / 'modelos/clasificador/metricas.json').read_text())
print('Elegido:', m['modelo'], '· umbral sugerido', m['umbral_sugerido'])
display(Image(filename=str(RAIZ / f"modelos/clasificador/matriz_{m['modelo']}_test.png"), width=560))

In [ ]:
# 8. Empaquetar y descargar resultados: modelos/clasificador (sin prueba/) + logs/clasificador (sin prueba/)
SALIDA = Path('/content/resultados_clasificador.zip')
with zipfile.ZipFile(SALIDA, 'w', zipfile.ZIP_DEFLATED) as z:
    for base in ('modelos/clasificador', 'logs/clasificador'):
        for p in (RAIZ / base).rglob('*'):
            rel = p.relative_to(RAIZ)
            if p.is_file() and 'prueba' not in rel.parts:
                z.write(p, rel.as_posix())
print(f'{SALIDA} ({SALIDA.stat().st_size / 1e6:.1f} MB)')
files.download(str(SALIDA))

**Compatibilidad de versiones.** El `.keras` guardado en Colab se carga en la PC con
`keras.models.load_model(..., compile=False)`. Si Colab tuviera una versión de Keras **más nueva**
que la del entorno local (`keras.__version__` de la celda 1 vs. la PC) y la carga fallara,
la salida es reentrenar localmente el modelo elegido con el mismo comando
(`python scripts/clasificador_entrenar.py --modelo <elegido>`).